**Main 2 Stage Pipeline for analyse Citation relevance**

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from dotenv import load_dotenv
from pathlib import Path
project_root = Path.cwd()
if not (project_root / "utils").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env", override=True)

True

In [3]:
from prediction.predict_main import *
from utils.citation_context import *
from utils.generate_report import *
import traceback as tr
from tqdm import tqdm

/Work/conda-envs/fire-mixsub/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MAX_REFERENCES = 32 # Prediction is in GPU, to quickly test take only few references
TEST_SET_MAIN_PAPERS = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/main-paper-set-1.txt"
FACET_PREDICTION_DIR = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/facet_predictions"

**Load Main papers from Paper Index**

In [15]:
paper_index = load_paper_index(max_files=4096) # turn off the max-files if you want to load all papers from dataset
all_main_paper_ids = list(paper_index.keys())
nb_main_papers = len(all_main_paper_ids)
print(f'Total {nb_main_papers} Main papers')

Loading paper index: 4095line [00:00, 6073.05line/s] 


Loaded 4096 papers from /Work/IISER/citation-data/SEKU-CRD-v1/paper_index_2024.jsonl (201.8 MB in memory)
Total 4096 Main papers


**Load Main Paper (M) IDs - This is Train Set**

We avoid these IDs for Test

In [6]:
with open(TEST_SET_MAIN_PAPERS) as f:
    test_set_main_paper_ids = [line.strip() for line in f if line.strip()]
Nm = len(test_set_main_paper_ids)
print(f"Loaded {Nm} main paper ids [To be excluded]")

Loaded 59 main paper ids [To be excluded]


**Find different Domains**

No Need to Run in main pipeline

In [ ]:
domains = set()
for m_id in all_main_paper_ids[:1000]:
    paper_data = get_paper_from_s3(m_id)
    paper_domain = paper_data['metadata']['discipline'] or None
    domains.add(paper_domain)
for d in domains:
    print(d)

### Prediction Block for Single M ###

In [16]:
citation_ctx_extractor = CitationContextExtractor()

**Main Model Loading [FACET]**

In [17]:
narrative_local_dir = os.environ.get(
        "NARRATIVE_MODEL_LOCAL_DIR", str(project_root / "model" / "narrative"))
context_classifier_local_dir = os.environ.get(
    "CONTEXT_CLASSIFIER_MODEL_LOCAL_DIR", str(project_root / "model" / "context_classifier"))

facet_predictor = CitationPipelinePredictor(
    narrative_local_dir=narrative_local_dir,
    context_classifier_local_dir=context_classifier_local_dir,
    narrative_checkpoint_kind="train-best",
    narrative_wandb_version="latest",
    narrative_version="v2",
    context_checkpoint_kind="train-best",
    context_wandb_version="latest",
)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/parthosarothi/.netrc.


Looking up 'qwen-narrative-adapter-train-best:latest' in W&B project 'psm-iiser-citation-relevancy'...

[Download] Downloading from S3...
  Bucket: psm-iiser
  Prefix: narrative-model/051026-123828/train_best
  Downloading: narrative-model/051026-123828/train_best/adapter_model/README.md -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/narrative/adapter_model/README.md
  Downloading: narrative-model/051026-123828/train_best/adapter_model/adapter_config.json -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/narrative/adapter_model/adapter_config.json
  Downloading: narrative-model/051026-123828/train_best/adapter_model/adapter_model.safetensors -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/narrative/adapter_model/adapter_model.safetensors
  Downloading: narrative-model/051026-123828/train_best/adapter_model/added_tokens.json -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/narrative/adapter_model/a

Loading weights: 100%|██████████| 339/339 [01:20<00:00,  4.19it/s]


✓ Loaded prompt from 'qwen-context-classifier-adapter-train-best:latest:v72' metadata (version=v1, output_keys=['FUNCTION', 'M_ASPECT', 'POLARITY', 'EVIDENCE', 'R_MATCH'])
Looking up 'qwen-context-classifier-adapter-train-best:latest' in W&B project 'psm-iiser-citation-relevancy'...

[Download] Downloading from S3...
  Bucket: psm-iiser
  Prefix: context-classifier/121026-082255/train_best
  Downloading: context-classifier/121026-082255/train_best/adapter_model/README.md -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/context_classifier/adapter_model/README.md
  Downloading: context-classifier/121026-082255/train_best/adapter_model/adapter_config.json -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/context_classifier/adapter_model/adapter_config.json
  Downloading: context-classifier/121026-082255/train_best/adapter_model/adapter_model.safetensors -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/model/context_classifier/adap

In [8]:
import random

**Pick a single random test paper**

In [ ]:
TARGET_DOMAINS = ["Physics","Mathematics"]
while True:
    rand_idx = random.randint(0,nb_main_papers-1)
    m_id = all_main_paper_ids[rand_idx]
    if m_id not in test_set_main_paper_ids:
        paper_data = get_paper_from_s3(m_id)
        paper_domain = paper_data['metadata']['discipline']
        if paper_domain in TARGET_DOMAINS:
            break
print(f"Random test main paper seletected {m_id}")

Random test main paper seletected 2303.11187
dict_keys(['paper_id', '_pdf_hash', '_source_hash', '_source_name', 'metadata', 'abstract', 'bib_entries', 'ref_entries', 'sections'])


**Pick a set of random test papers**

In [19]:
T = 25
TARGET_DOMAINS = ["Physics","Mathematics"]
test_set = []
while len(test_set) < T:
    rand_idx = random.randint(0,nb_main_papers-1)
    m_id = all_main_paper_ids[rand_idx]
    if m_id not in test_set_main_paper_ids:
        paper_data = get_paper_from_s3(m_id)
        paper_domain = paper_data['metadata']['discipline']
        if paper_domain in TARGET_DOMAINS:
            test_set.append((m_id,paper_domain))
            print(f"Random test main paper seletected {m_id}:{paper_domain}")
with open(f"/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/M_test_set_141026.txt","w") as f:
    for m_id,p_dom in test_set:
        f.write(f"{m_id}:{p_dom}\n")
f.close()

Random test main paper seletected 2307.14625:Physics
Random test main paper seletected 2311.00644:Physics
Random test main paper seletected 2306.08156:Physics
Random test main paper seletected hep-th/0404239:Physics
Random test main paper seletected 2304.07698:Physics
Random test main paper seletected 2305.03420:Physics
Random test main paper seletected 0807.2577:Physics
Random test main paper seletected 0807.2838:Physics
Random test main paper seletected 2307.16712:Physics
Random test main paper seletected 2312.11946:Physics
Random test main paper seletected 2305.04663:Physics
Random test main paper seletected 2305.15350:Physics
Random test main paper seletected 2305.10422:Physics
Random test main paper seletected 2310.08178:Physics
Random test main paper seletected 0912.0029:Physics
Random test main paper seletected 2307.09680:Physics
Random test main paper seletected 0912.0707:Physics
Random test main paper seletected 2307.16308:Physics
Random test main paper seletected 2302.10930:P

**Get Main Paper content from S3**

In [31]:
m_id = test_set[5]
print(f"Starting test for {m_id}")
m_data = paper_index[m_id] # main paper data from paper index file
m_title = m_data["title"]
print(f'Main Title: {m_title}')
main_paper_data = get_paper_from_s3(m_id)
print("Retrieved main paper data from S3...Extracting titles (not needed) and abstract")
_,m_abstract = get_title_and_abstract(main_paper_data)
print("Retrieved abstract...Extracting Aspects/Narrative from abstract")

Starting test for 2303.06679
Main Title: RotoGBML: Towards Out-of-Distribution Generalization for Gradient-Based
  Meta-Learning
Retrieved main paper data from S3...Extracting titles (not needed) and abstract
Retrieved abstract...Extracting Aspects/Narrative from abstract


In [32]:
m_abstract

'Gradient-based meta-learning (GBML) algorithms are able to fast adapt to new\ntasks by transferring the learned meta-knowledge, while assuming that all tasks\ncome from the same distribution (in-distribution, ID). However, in the real\nworld, they often suffer from an out-of-distribution (OOD) generalization\nproblem, where tasks come from different distributions. OOD exacerbates\ninconsistencies in magnitudes and directions of task gradients, which brings\nchallenges for GBML to optimize the meta-knowledge by minimizing the sum of\ntask gradients in each minibatch. To address this problem, we propose RotoGBML,\na novel approach to homogenize OOD task gradients. RotoGBML uses reweighted\nvectors to dynamically balance diverse magnitudes to a common scale and uses\nrotation matrixes to rotate conflicting directions close to each other. To\nreduce overhead, we homogenize gradients with the features rather than the\nnetwork parameters. On this basis, to avoid the intervention of non-caus

**Additional Prediction for Main paper**

In [33]:
m_narrative = facet_predictor.predict_narrative(m_abstract,max_new_tokens=2048)
print("Gathering all citation context")
status,cite_ctx = find_citation_context(m_id,bulk_paper_index=paper_index,citation_context_extractor=citation_ctx_extractor)
ref_contexts = cite_ctx[m_id]["references"] # all citation contexts

Gathering all citation context
Extracted context for 2303.06679 success [27/28]


**Predict on Reference Paper Data [Batch]**

In [34]:
import time

t0 = time.time()
in_corpus_refs_ids = m_data["in_corpus_references"] # this comes from paper index data
in_corpus_refs_ids = in_corpus_refs_ids[:min(len(in_corpus_refs_ids),MAX_REFERENCES)]
ref_papers_data = await get_papers_from_s3(in_corpus_refs_ids)

references = await facet_predictor.predict_references(
    m_title=m_title, m_narrative=m_narrative,
    ref_contexts=ref_contexts, ref_papers_data=ref_papers_data,
    batch_size=8, narrative_max_new_tokens=2048, relation_max_new_tokens=512,
)
facet_relevance_batched = {"m_id":m_id,"m_title":m_title,"m_abstract":m_abstract,"m_aspects":m_narrative,"references":references}

elapsed = time.time() - t0
print(f"Batched pipeline: {len(references)} references -- wall time {elapsed:.1f}s")

with open(f"{FACET_PREDICTION_DIR}/{m_id}.json","w") as f:
    json.dump(facet_relevance_batched,f,indent=3)

# generate report
out_path = f"{FACET_PREDICTION_DIR}/reports/{m_id}_report.md"
with open(out_path, "w") as f:
    f.write(generate_report(facet_relevance_batched))
print(f"Wrote report -> {out_path}")

Fetching papers: 100%|██████████| 28/28 [00:01<00:00, 21.70it/s]


26/27 references usable for batching


Classifying citation relations: 100%|██████████| 40/40 [00:43<00:00,  1.10s/context]

Batched pipeline: 26 references -- wall time 149.9s
Wrote report -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/data/facet_predictions/reports/2303.06679_report.md


### Embedding Computation ##

**Unload Predictor Model**

In [35]:
import gc
import torch

print(f"GPU memory before unload: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
      f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

# facet_predictor.narrative_model.model IS facet_predictor.context_model.model
# (one shared base + both LoRA adapters -- see CitationPipelinePredictor), so
# dropping the single top-level reference is enough; nothing else in this
# notebook holds a separate reference to the model/tokenizer.
del facet_predictor
gc.collect()
torch.cuda.empty_cache()

print(f"GPU memory after unload : {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
      f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

GPU memory before unload: 5.69 GB allocated, 11.64 GB reserved
GPU memory after unload : 0.09 GB allocated, 5.58 GB reserved


In [5]:
from prediction.qwen_embedding import *

In [6]:
embedding_qwen = AspectSimilarity(model_name="Qwen/Qwen3-Embedding-4B")

The `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead.
Loading weights: 100%|██████████| 398/398 [00:03<00:00, 118.02it/s]


In [15]:
FACET_PREDICTION_FILE = "/Work/Experiments/seku-crd-citation-processor/citation-ranker/data/facet_predictions/2311.03695.json"
embedding_qwen.run(input_file=FACET_PREDICTION_FILE)

aspect similarity: 100%|██████████| 1/1 [00:02<00:00,  2.59s/it]


In [16]:
# embedding_qwen.run() just overwrote FACET_PREDICTION_FILE in place with
# aspect_similarity added -- if a report was already generated for it, that
# report is now stale, so regenerate it from the updated JSON too.
report_path = f"{FACET_PREDICTION_DIR}/reports/{Path(FACET_PREDICTION_FILE).stem}_report.md"
if os.path.exists(report_path):
    with open(FACET_PREDICTION_FILE) as f:
        updated_facet_relevance = json.load(f)
    with open(report_path, "w") as f:
        f.write(generate_report(updated_facet_relevance))
    print(f"Overwrote existing report -> {report_path}")
else:
    print(f"No existing report at {report_path}; nothing to overwrite")

Overwrote existing report -> /Work/Experiments/seku-crd-citation-processor/citation-ranker/data/facet_predictions/reports/2311.03695_report.md
